<!-- dd:dd-lesson-tr-1 -->

# Rays as tensors (ARENA 0.1)

*PyTorch · `tr-1`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "tr-1"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_preflight_torch():
    """Import torch once, here, where a failure can still be explained.

    Every drill cell opens with `import torch as t`, so the learner meets a
    broken torch install as a traceback through torch's own internals — the one
    reported was `AttributeError: partially initialized module 'torch' has no
    attribute 'fx'` from `torch/_export/utils.py`, raised while evaluating a
    function's annotations. That message names neither the cause nor the cure,
    and it is not even the real error: it is what a LATER import sees after an
    earlier one died partway and left the half-built module in `sys.modules`.
    Python does unwind a failed import normally, but a torch that was swapped
    on disk under a running kernel (a `pip install` mid-session) or shadowed by
    a stray `torch.py` gets far enough in to be cached before it falls over.

    So: purge the wreckage and retry ONCE, which is the whole fix whenever the
    first failure was transient, and report what actually broke when it is not.
    Importing torch in this cell rather than lazily is safe now in a way the
    `_dd_tensor` comment below still guards against for the per-comparison
    path — the bank is 448/448 torch and every notebook imports it a few cells
    down, so there is no numpy-only notebook left to charge for it.

    Never raises: a checker that refuses to load over this would take the
    lesson down with the runtime.
    """

    def _purge():
        # Submodules too, and that is the whole point. Python drops only the
        # module that raised, so `torch` goes and a `torch._export` imported
        # seconds earlier STAYS — and the next `import torch` re-runs
        # `torch/__init__.py` straight back into that stale submodule, which
        # reaches for a `torch.fx` the half-built parent has not bound yet.
        # Leaving one behind reproduces the bug instead of clearing it.
        for name in [n for n in sys.modules if n == "torch" or n.startswith("torch.")]:
            del sys.modules[name]

    def _usable(mod):
        # `import torch` does NOT re-execute a module already in sys.modules,
        # so a corpse left by a failed import is imported "successfully" and
        # the error surfaces later, from the learner's own cell. Judge the
        # object, not the statement: a torch that finished has both of these.
        return hasattr(mod, "fx") and hasattr(mod, "__version__")

    cached = sys.modules.get("torch")
    if cached is not None and not _usable(cached):
        _purge()

    for attempt in (1, 2):
        try:
            import torch
            if not _usable(torch):
                raise ImportError(
                    "torch imported but is only partially initialised "
                    "(no .fx) — an earlier import in this session died partway"
                )
            return True
        except Exception as exc:
            if attempt == 1:
                _purge()
                continue
            print(
                "⚠️  This runtime cannot import PyTorch, so no drill in this "
                "notebook will run.\n"
                "    %s: %s\n"
                "    Fix: Runtime ▸ Disconnect and delete runtime, then reopen "
                "this notebook and run\n"
                "    this cell first. If it comes back, check for a file named "
                "torch.py in /content,\n"
                "    and re-run any pip install BEFORE anything imports torch."
                % (type(exc).__name__, exc)
            )
    return False


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_preflight_torch()
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNrtW9tu4zYQ/RXBTzFgBeSQ9KUf0C/om2osgqyLFkg3QewtCgT590okRVG8yyTtPDQIHImmOPfRmSHzsdod9qtfmo/VHz/6"
    "P6vz68s/p9WmWT0/nU/nfqT7WJ1Pl59v355fv5+GGX/9/fb6fmkur+/PfzZP5+by+w8x/+VFLfBANg16RJsG84/1MOH079vp"
    "+XL6/q2/eB9mdh2fMn4cN804wOYDeJxxXH1umqv4oZumxdNaPoZabHLUokcifjChW4bxnposh7+fr3i9BLBp2LDKdviAuEbZ"
    "TKPiwYH8sWegt/mhjs2DfHFVgMlXtmHT3MvhUjB8kCzyOExe0lN63yNUNdZimtAckg3XIJ/L88sWpDaD5FvQ6EO20YdV6GPU"
    "4WYUR4mJuCZyDc08uIJ53Cw+aMlO8tm5UqHussfj+no7BdnwcpBJlUWEB6by/S6sCGBOrQzPWcxKY8JdY23uflLIWdQVeCWg"
    "lDAYJylmaMEXajTs9ejDpm7mqhhjEsYHtcgk5Y3ZdYeB0EFQdl73N6BDmbCsB01Ka6WrVd11E3LBgicswQALgoHpMZbJgCMe"
    "/df9DZF0SRz+Ye1pHRsQc1XlCbS8JwBneRviteei/+Uu2gdQr9IMhRK+GITcqWeFiy9p8r95MGVYEgdIYl3HrIqOFQT38eCp"
    "CFqs+ZZv0jQnzzDXsYgSOFSFTWUpsMzwdLEUVKcuXaFOpRDOplYZFhnIha8sqaQaJs4NzkeyDHWIZ8iDlQIxqhKeWMPKIgEn"
    "F+6l6imRGTsWoD0Vtlp1db0BEF+kd4B2QGd+gccZkxFwvcBoAwlITVB6t2/yNCLSFLCgMjgiz4q7WI6iGmylVqmG4WalWjRX"
    "a/glNx8l9soygHspGTWgZqYnUgeh+VylB6jELCHG1yyZUllGTvLH44PWNOT9QvfNdDd9ZrAkotTL0QxoiLQ2moZWSVqQWBFb"
    "noVdA1lZRZRGKY0wlF8ZEV43hLK1ZXSXI7jGpmhiNSpfX+WGuAZ3foF2zg5RseJ2fI35B0T5K3BRvN2M5yW5qsuhkMcJPCIU"
    "F+2GaEbdVkiR3l6bLCKHgjMn6XjX54vnZNgQ5xrzkn+hQkA148Jql6sI8Zt43swqEhLu0jDwLjQafAV6PnNNTO28I//Cw8X8"
    "YWUyfB+TGdtPy6oafTuyVOfWUeEaAFK9uGOtPVSotefPth2OlYPY3N0jChjOMDtADftbqgSzIeMFBP3M4hECKWFKKhDWPd4b"
    "mWggLK1BbheNSUrZ5isFu/OVpOwtNts8e4BZGIGZr7c+ylSzB72PPUbsqcbcnP769HI+VY0WoTYaZuO395+34IIlKUMajn0x"
    "JPJg76vNB9Yl8br15goUy9qukCaUGliXtCyx8AoNMCb2HLnz7SVrxtCEPLf3sTf5usBFz3WwrKtfGMXoAoNm+/TGmjTyroaR"
    "U8CqD3lPk4JViCf22yJ69kT/1AgI1iba0zSXERRxJr8i0bLNxGBrCztBR2C50b1Itd2D9C0/nLK3mqIyXGrnEoKO7ZMAEriD"
    "MvuqbDl3zN5TJfC/8T2n+thyCRalzpCYLEFMVsLH0XIxl0kweRq5tacZfuXduy9z/o3GGKEm3QJn31gwSxk5yQ61uErspKXs"
    "Sb9I5kDTRkVe5tCgw21DCo1xlSABO2af8riKu8M8mOucyuL0DByXwC1OeY/gwu+I8YBE4jkVY5qHXfcklDCn0Gktvq75zyde"
    "mVocg0GTx2xvmy4e5GYwWW+aoVFjtZMrbXlzJZpkM4jRUHOFf+uXUQkqTbCrd0xoEVaGFMAHCWkbCqL96Gknm3ybhFzbkvEp"
    "T5SwJacph/NixBGR+3u4QzvZFXJ2BUHhJxaklEWEyYSII+K0M/wwEj1+fv4HvOphiQ=="
)
print("Delta Drills checker ready — 32 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-torch-out-argument -->

## Filling a tensor in place with out=

`torch.out-argument`


### out= writes into storage you already own


Every constructor you know so far **allocates** a fresh tensor and hands it
back. `t.linspace(0, 1, 5)` makes five new floats somewhere in memory. Often
that is exactly right. But when you have already built the tensor that will
hold the result — a canvas of zeros with the right shape — a fresh allocation
is one extra step: make the values, then copy them into place.

`out=` removes that step. Pass the **target** as `out=` and the function writes
its result **directly into that tensor's storage** instead of allocating:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']("import torch as t\n\ncanvas = t.zeros((5, 3))\nt.linspace(0, 1, 5, out=canvas[:, 1])\nprint(canvas)\n# Hidden checks\nassert _delta_output == 'tensor([[0.0000, 0.0000, 0.0000],\\n        [0.0000, 0.2500, 0.0000],\\n        [0.0000, 0.5000, 0.0000],\\n        [0.0000, 0.7500, 0.0000],\\n        [0.0000, 1.0000, 0.0000]])\\n'\n", globals()), end='')




Three things to notice.

- The target was a **slice**, `canvas[:, 1]`. A slice is a view of the
  canvas, so writing into the view wrote into the canvas — column 1 is
  filled and the other columns are untouched.
- The **return value is the target itself**, not a new tensor. Catching it in
  a name gives you the same view back:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']("import torch as t\n\ncanvas = t.zeros(4)\nview = canvas[1:3]\nfilled = t.arange(2, out=view)\nprint(filled is view)\nprint(canvas)\n# Hidden checks\nassert _delta_output == 'True\\ntensor([0., 0., 1., 0.])\\n'\n", globals()), end='')




- The target's **shape must already match** what the call would produce.
  `t.linspace(0, 1, 5)` is five values, so the target must hold five. If it
  does not, PyTorch does **not** raise: it warns, **resizes the target** to a
  flat run of the canvas's own storage, and writes the values there — which
  is *not* the column you named. The numbers land in the wrong slots of the
  canvas, silently. Count the slots of the slice before you call:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\ncanvas = t.zeros((5, 3))\nt.linspace(0, 1, 4, out=canvas[:, 1])   # 4 values into a 5-slot column\nprint(canvas)                            # the values sit across row 0..1, not down column 1\n# Hidden checks\nassert canvas.shape == (5, 3)\nassert t.allclose(canvas.flatten()[1:5], t.tensor([0., 1/3, 2/3, 1.]))\nassert canvas.flatten()[5:].count_nonzero().item() == 0\nassert canvas.flatten()[0].item() == 0\n', globals()), end='')




`out=` works the same on `t.arange`, `t.linspace`, and most other tensor
producing functions. The dtype follows the target: `t.arange(3, out=floats)`
writes `0., 1., 2.` because the target is float.


Build five y-coordinates as one column of a zero canvas, in one call:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\npoints = t.zeros((5, 3))\nt.linspace(-1.0, 1.0, 5, out=points[:, 1])   # column 1 = y\nprint(points)\n# Hidden checks\nassert points[0, 1] == -1.0 and points[-1, 1] == 1.0\nassert points[:, 0].sum() == 0.0            # other columns untouched\n', globals()), end='')




Input → output of the same idea, on a shorter canvas:

| call | canvas afterwards |
| --- | --- |
| `t.linspace(0, 1, 3, out=c[:, 0])` on `c = t.zeros((3, 2))` | `[[0, 0], [0.5, 0], [1, 0]]` |
| `t.arange(3, out=c[:, 1])` on the same `c` | `[[0, 0], [0.5, 1], [1, 2]]` |


<!-- dd:dd-q798 -->

### Problem 798 · faded — your turn

One call writes the column; the canvas is the thing you return.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0.0, 0.0, 0.0], [0.0, 0.5, 0.0], [0.0, 1.0, 0.0]]
```


In [ ]:
def solve(n, lo, hi, col):
    """n evenly spaced values from lo to hi, written into column col."""
    canvas = t.zeros((n, 3))
    t._____(lo, hi, n, _____=canvas[:, col])
    return canvas.tolist()


import torch as t
# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 0.0, 1.0, 1)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(798)


In [ ]:
#@title 💡 Solution — Problem 798
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(n, lo, hi, col):
    """n evenly spaced values from lo to hi, written into column col."""
    canvas = t.zeros((n, 3))
    t.linspace(lo, hi, n, out=canvas[:, col])
    return canvas.tolist()


example = (3, 0.0, 1.0, 1)
print(solve(*example))


Build five y-coordinates as one column of a zero canvas, in one call:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\npoints = t.zeros((5, 3))\nt.linspace(-1.0, 1.0, 5, out=points[:, 1])   # column 1 = y\nprint(points)\n# Hidden checks\nassert points[0, 1] == -1.0 and points[-1, 1] == 1.0\nassert points[:, 0].sum() == 0.0            # other columns untouched\n', globals()), end='')




Input → output of the same idea, on a shorter canvas:

| call | canvas afterwards |
| --- | --- |
| `t.linspace(0, 1, 3, out=c[:, 0])` on `c = t.zeros((3, 2))` | `[[0, 0], [0.5, 0], [1, 0]]` |
| `t.arange(3, out=c[:, 1])` on the same `c` | `[[0, 0], [0.5, 1], [1, 2]]` |


<!-- dd:dd-q799 -->

### Problem 799 · faded — your turn

The target is float, so the integers land as 0., 1., 2.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0.0, 1.0, 2.0], [0.0, 0.0, 0.0]]
```


In [ ]:
def solve(n, row):
    """The integers 0..n-1 written into row `row` of a 2×n canvas."""
    canvas = t.zeros((2, n))
    t._____(n, _____=canvas[row])
    return canvas.tolist()


import torch as t
# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(799)


In [ ]:
#@title 💡 Solution — Problem 799
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(n, row):
    """The integers 0..n-1 written into row `row` of a 2×n canvas."""
    canvas = t.zeros((2, n))
    t.arange(n, out=canvas[row])
    return canvas.tolist()


example = (3, 0)
print(solve(*example))


<!-- dd:dd-q800 -->

### Problem 800 · independent

Return an n×2 tensor as a nested list whose column 0 is `0..n-1` and whose column 1 runs from `lo` to `hi` in n steps. Build one zero canvas and fill both columns with `out=`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0.0, 0.0], [1.0, 0.5], [2.0, 1.0]]
```


In [ ]:
import torch as t


def solve(n, lo, hi):
    """Column 0 = arange, column 1 = linspace(lo, hi), one canvas."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 0.0, 1.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(800)


In [ ]:
#@title 💡 Solution — Problem 800
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(n, lo, hi):
    """Column 0 = arange, column 1 = linspace(lo, hi), one canvas."""
    canvas = t.zeros((n, 2))
    t.arange(n, out=canvas[:, 0])
    t.linspace(lo, hi, n, out=canvas[:, 1])
    return canvas.tolist()


example = (3, 0.0, 1.0)
print(solve(*example))


<!-- dd:dd-q801 -->

### Problem 801 · independent

Return a tuple `(filled, canvas)`: fill `canvas[:, 1]` of an n×2 zero canvas with `linspace(0, 1, n, out=...)`, keep the RETURNED tensor as `filled`, and return both as lists. `filled` is the column view, so its list matches column 1 of the canvas.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0.0, 0.5, 1.0], [[0.0, 0.0], [0.0, 0.5], [0.0, 1.0]])
```


In [ ]:
import torch as t


def solve(n):
    """out= hands back the target view; the canvas shows the write."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(801)


In [ ]:
#@title 💡 Solution — Problem 801
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(n):
    """out= hands back the target view; the canvas shows the write."""
    canvas = t.zeros((n, 2))
    filled = t.linspace(0, 1, n, out=canvas[:, 1])
    return (filled.tolist(), canvas.tolist())


example = (3,)
print(solve(*example))


<!-- dd:dd-q802 -->

### Problem 802 · independent

Fill every column of an n×3 zero canvas: column 0 with `arange`, column 1 with `linspace(lo, hi)`, column 2 with `linspace(hi, lo)` (reversed). Return the canvas as a nested list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0.0, 0.0, 1.0], [1.0, 0.5, 0.5], [2.0, 1.0, 0.0]]
```


In [ ]:
import torch as t


def solve(n, lo, hi):
    """Three out= writes, three columns."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 0.0, 1.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(802)


In [ ]:
#@title 💡 Solution — Problem 802
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(n, lo, hi):
    """Three out= writes, three columns."""
    canvas = t.zeros((n, 3))
    t.arange(n, out=canvas[:, 0])
    t.linspace(lo, hi, n, out=canvas[:, 1])
    t.linspace(hi, lo, n, out=canvas[:, 2])
    return canvas.tolist()


example = (3, 0.0, 1.0)
print(solve(*example))


<!-- dd:dd-q803 -->

### Problem 803 · independent

Given a canvas as a nested list, write `linspace(lo, hi, k)` into its first `k` rows of column `col` using `out=` on a slice, and return the whole canvas as a nested list. Rows after `k` stay as they were.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0.0, 9.0], [1.0, 9.0], [9.0, 9.0]]
```


In [ ]:
import torch as t


def solve(canvas, k, lo, hi, col):
    """linspace into the first k rows of one column of an existing canvas."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[9.0, 9.0], [9.0, 9.0], [9.0, 9.0]], 2, 0.0, 1.0, 0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(803)


In [ ]:
#@title 💡 Solution — Problem 803
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(canvas, k, lo, hi, col):
    """linspace into the first k rows of one column of an existing canvas."""
    canvas = t.tensor(canvas, dtype=t.float32)
    t.linspace(lo, hi, k, out=canvas[:k, col])
    return canvas.tolist()


example = ([[9.0, 9.0], [9.0, 9.0], [9.0, 9.0]], 2, 0.0, 1.0, 0)
print(solve(*example))


<!-- dd:dd-q804 -->

### Problem 804 · independent

Return the integers `start..stop-1` written into row `row` of an m×(stop-start) canvas of zeros with dtype int64 (`t.zeros(..., dtype=t.int64)`), using `arange(start, stop, out=...)`. Return the canvas as a nested list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0, 0, 0], [3, 4, 5]]
```


In [ ]:
import torch as t


def solve(m, start, stop, row):
    """arange(start, stop) into an integer canvas row."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, 3, 6, 1)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(804)


In [ ]:
#@title 💡 Solution — Problem 804
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(m, start, stop, row):
    """arange(start, stop) into an integer canvas row."""
    canvas = t.zeros((m, stop - start), dtype=t.int64)
    t.arange(start, stop, out=canvas[row])
    return canvas.tolist()


example = (2, 3, 6, 1)
print(solve(*example))


<!-- dd:dd-q805 -->

### Problem 805 · independent

A 3-D canvas of zeros shaped (n, 2, 3). Fill `canvas[:, 1, 1]` (the y of the second row of every slab) with `linspace(lo, hi, n)` via `out=`. Return the canvas as a nested list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[[0.0, 0.0, 0.0], [0.0, -1.0, 0.0]], [[0.0, 0.0, 0.0], [0.0, 1.0, 0.0]]]
```


In [ ]:
import torch as t


def solve(n, lo, hi):
    """One column through a 3-D canvas, written with out=."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, -1.0, 1.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(805)


In [ ]:
#@title 💡 Solution — Problem 805
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(n, lo, hi):
    """One column through a 3-D canvas, written with out=."""
    canvas = t.zeros((n, 2, 3))
    t.linspace(lo, hi, n, out=canvas[:, 1, 1])
    return canvas.tolist()


example = (2, -1.0, 1.0)
print(solve(*example))


#### Common mistakes

- **"`out=` is just a faster way to get a return value."** The return value
  is the *target*. What changes is that the target's storage is written. If
  you throw the return value away, the write still happened.
- **"A shape mismatch raises."** It does not. PyTorch warns, resizes the
  target over the canvas's storage and writes the values into the wrong
  slots — a corrupted canvas, not an error. Check the slice's slot count
  against the number of values you pass.
- **"Row 0 is column 0."** `canvas[0]` is a row; `canvas[:, 0]` is a column.
  Both are valid `out=` targets and both are the wrong one half the time.


<!-- dd:dd-kp-torch-slice-assignment -->

## Writing tensors into slices: broadcasting on the left

`torch.slice-assignment`


### `x[sel] = value` writes in place, broadcasting the value


You have met `x[2:5] = 0.0`: assigning to a slice writes through the view into
`x`. That page used a scalar. Here the right-hand side becomes a **tensor**,
the slice becomes **multi-axis**, and two rules make this the workhorse of
tensor building:

1. **The right-hand side broadcasts to the slice.** A scalar fills every slot;
   a 1-D tensor of the right length fills a column or a row; a full tensor of
   the slice's shape is copied element by element. Ordinary broadcasting
   rules, applied to the *slice's* shape.
2. **The write is in place.** Every other view of the same storage sees it,
   and the tensor's shape and dtype never change (a float written into an int
   tensor is truncated, not upcast).


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']("import torch as t\n\nx = t.zeros((3, 4))\nx[:, 1] = 3.0                 # scalar → whole column\nx[0] = t.arange(4)            # 1-D of length 4 → one row\nx[1:, 2:] = t.ones((2, 2))    # block of matching shape\nprint(x)\n# Hidden checks\nassert _delta_output == 'tensor([[0., 1., 2., 3.],\\n        [0., 3., 1., 1.],\\n        [0., 3., 1., 1.]])\\n'\n", globals()), end='')




The same write has a **method spelling** you will see in library code:
`x[0].copy_(src)` is `x[0] = src`, with `src` broadcast to the slice. The
trailing underscore is PyTorch's mark for *in place* — it acts on the view,
so it writes into `x`:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']("import torch as t\n\nx = t.zeros((2, 3))\nx[1].copy_(t.tensor([1.0, 2.0, 3.0]))\nx[:, 0].copy_(t.tensor(7.0))        # a 0-d tensor broadcasts like a scalar\nprint(x)\n# Hidden checks\nassert _delta_output == 'tensor([[7., 0., 0.],\\n        [7., 2., 3.]])\\n'\n", globals()), end='')




Shape still has to work. `x[:, 1] = t.arange(5)` on a 3-row `x` is a
`RuntimeError` — five values cannot broadcast to three slots — and, unlike
`out=`, assignment **does** raise, which is why it is the safer of the two.


Three slabs of shape (2, 3). Give row 1 of every slab an x of 1 and a y equal
to the slab's own index, then scale the middle slab's row 1 by 10 — three
writes, one canvas:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nslabs = t.zeros((3, 2, 3))\nslabs[:, 1, 0] = 1.0                 # scalar broadcast to 3 slots\nslabs[:, 1, 1] = t.arange(3)         # 1-D of length 3 into 3 slots\nslabs[1, 1] = slabs[1, 1] * 10       # a whole row, rewritten from itself\nprint(slabs)\n# Hidden checks\nassert slabs[1, 1].tolist() == [10.0, 10.0, 0.0]\n', globals()), end='')




| write | slice shape | RHS shape | result |
| --- | --- | --- | --- |
| `x[:, 0] = 1.0` on 3×2 | `(3,)` | `()` | column 0 all ones |
| `x[0] = t.arange(2)` on 3×2 | `(2,)` | `(2,)` | row 0 = `[0, 1]` |
| `x[:, 0] = t.arange(2)` on 3×2 | `(3,)` | `(2,)` | `RuntimeError` |


<!-- dd:dd-q809 -->

### Problem 809 · faded — your turn

The slice is the target; the scalar broadcasts to every slot of it.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0]]
```


In [ ]:
def solve(n, v, col):
    """One column set to v by assigning into the slice."""
    canvas = t.zeros((n, 3))
    canvas[_____, _____] = v
    return canvas.tolist()


import torch as t
# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 1.0, 0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(809)


In [ ]:
#@title 💡 Solution — Problem 809
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(n, v, col):
    """One column set to v by assigning into the slice."""
    canvas = t.zeros((n, 3))
    canvas[:, col] = v
    return canvas.tolist()


example = (3, 1.0, 0)
print(solve(*example))


Three slabs of shape (2, 3). Give row 1 of every slab an x of 1 and a y equal
to the slab's own index, then scale the middle slab's row 1 by 10 — three
writes, one canvas:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nslabs = t.zeros((3, 2, 3))\nslabs[:, 1, 0] = 1.0                 # scalar broadcast to 3 slots\nslabs[:, 1, 1] = t.arange(3)         # 1-D of length 3 into 3 slots\nslabs[1, 1] = slabs[1, 1] * 10       # a whole row, rewritten from itself\nprint(slabs)\n# Hidden checks\nassert slabs[1, 1].tolist() == [10.0, 10.0, 0.0]\n', globals()), end='')




| write | slice shape | RHS shape | result |
| --- | --- | --- | --- |
| `x[:, 0] = 1.0` on 3×2 | `(3,)` | `()` | column 0 all ones |
| `x[0] = t.arange(2)` on 3×2 | `(2,)` | `(2,)` | row 0 = `[0, 1]` |
| `x[:, 0] = t.arange(2)` on 3×2 | `(3,)` | `(2,)` | `RuntimeError` |


<!-- dd:dd-q810 -->

### Problem 810 · faded — your turn

copy_ takes a tensor; wrap the list first.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0.0, 0.0, 0.0], [1.0, 2.0, 3.0]]
```


In [ ]:
def solve(m, row, row_values):
    """One row overwritten with copy_."""
    canvas = t.zeros((m, len(row_values)))
    canvas[row]._____(t.tensor(row_values))
    return canvas.tolist()


import torch as t
# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, 1, [1.0, 2.0, 3.0])
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(810)


In [ ]:
#@title 💡 Solution — Problem 810
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(m, row, row_values):
    """One row overwritten with copy_."""
    canvas = t.zeros((m, len(row_values)))
    canvas[row].copy_(t.tensor(row_values))
    return canvas.tolist()


example = (2, 1, [1.0, 2.0, 3.0])
print(solve(*example))


<!-- dd:dd-q811 -->

### Problem 811 · independent

Return an n×2 canvas as a nested list with column 0 all equal to `a` and column 1 all equal to `b`, built from `t.zeros` and two slice assignments.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[1.0, -1.0], [1.0, -1.0], [1.0, -1.0]]
```


In [ ]:
import torch as t


def solve(n, a, b):
    """Two scalar broadcasts into two columns."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 1.0, -1.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(811)


In [ ]:
#@title 💡 Solution — Problem 811
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(n, a, b):
    """Two scalar broadcasts into two columns."""
    canvas = t.zeros((n, 2))
    canvas[:, 0] = a
    canvas[:, 1] = b
    return canvas.tolist()


example = (3, 1.0, -1.0)
print(solve(*example))


<!-- dd:dd-q812 -->

### Problem 812 · independent

Return an n×3 canvas as a nested list whose column 1 holds `0..n-1` (assign `t.arange(n)` into the column slice) and whose other columns are zero.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 2.0, 0.0]]
```


In [ ]:
import torch as t


def solve(n):
    """A 1-D tensor assigned into a column."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(812)


In [ ]:
#@title 💡 Solution — Problem 812
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(n):
    """A 1-D tensor assigned into a column."""
    canvas = t.zeros((n, 3))
    canvas[:, 1] = t.arange(n)
    return canvas.tolist()


example = (3,)
print(solve(*example))


<!-- dd:dd-q813 -->

### Problem 813 · independent

Two views of one storage. Take `col = canvas[:, 0]` of an n×2 zero canvas, then assign `v` to `col[:]`. Return a tuple `(canvas.tolist(), col.tolist())` — both should show the write.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([[3.0, 0.0], [3.0, 0.0]], [3.0, 3.0])
```


In [ ]:
import torch as t


def solve(n, v):
    """A write through one view shows in every view."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, 3.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(813)


In [ ]:
#@title 💡 Solution — Problem 813
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(n, v):
    """A write through one view shows in every view."""
    canvas = t.zeros((n, 2))
    col = canvas[:, 0]
    col[:] = v
    return (canvas.tolist(), col.tolist())


example = (2, 3.0)
print(solve(*example))


<!-- dd:dd-q814 -->

### Problem 814 · independent

Set the lower-right `k×k` block of an n×n zero canvas to `v` with one slice assignment. Return the canvas as a nested list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0.0, 0.0, 0.0], [0.0, 1.0, 1.0], [0.0, 1.0, 1.0]]
```


In [ ]:
import torch as t


def solve(n, k, v):
    """A block write via two slices."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 2, 1.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(814)


In [ ]:
#@title 💡 Solution — Problem 814
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(n, k, v):
    """A block write via two slices."""
    canvas = t.zeros((n, n))
    canvas[n - k:, n - k:] = v
    return canvas.tolist()


example = (3, 2, 1.0)
print(solve(*example))


<!-- dd:dd-q815 -->

### Problem 815 · independent

Write into an existing canvas, given as a nested list of floats: row `row` becomes `value` everywhere (slice assignment), and column `col` becomes `0..n-1` via `copy_` of `t.arange(n)` (the column write happens LAST, so it wins at the crossing). Return the canvas as a nested list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[7.0, 0.0], [0.0, 1.0]]
```


In [ ]:
import torch as t


def solve(canvas, row, col, value):
    """Assign a row, then copy_ a column; order matters at the crossing."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[0.0, 0.0], [0.0, 0.0]], 0, 1, 7.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(815)


In [ ]:
#@title 💡 Solution — Problem 815
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(canvas, row, col, value):
    """Assign a row, then copy_ a column; order matters at the crossing."""
    canvas = t.tensor(canvas)
    n = canvas.shape[0]
    canvas[row] = value
    canvas[:, col].copy_(t.arange(n))
    return canvas.tolist()


example = ([[0.0, 0.0], [0.0, 0.0]], 0, 1, 7.0)
print(solve(*example))


<!-- dd:dd-q816 -->

### Problem 816 · independent

Return `(before, after)`: assign the float `2.7` into slot 0 of `t.zeros(n, dtype=t.int64)` and report the tensor as a list before and after. The dtype never changes on assignment.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([0, 0], [2, 0])
```


In [ ]:
import torch as t


def solve(n):
    """A float assigned into an int tensor is truncated, not upcast."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2,)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(816)


In [ ]:
#@title 💡 Solution — Problem 816
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(n):
    """A float assigned into an int tensor is truncated, not upcast."""
    canvas = t.zeros(n, dtype=t.int64)
    before = canvas.tolist()
    canvas[0] = 2.7
    return (before, canvas.tolist())


example = (2,)
print(solve(*example))


#### Common mistakes

- **"`x[:, 0] = v` makes a new tensor."** It does not; it writes into the
  storage `x` already has, and every view of that storage sees the change.
  Rebinding (`x = ...`) is what makes new tensors.
- **"Rows and columns are interchangeable in a write."** `x[0]` is a row,
  `x[:, 0]` a column. The shapes differ, so a 1-D right-hand side that fits
  one will raise on the other whenever the tensor is not square.
- **"Assignment upcasts."** The tensor's dtype is fixed; a float written into
  an int tensor is truncated toward zero.


<!-- dd:dd-kp-raytracing-ray-parametrisation -->

## A ray is an origin and a direction

`raytracing.ray-parametrisation`


### origin + u · direction, and the (2, 3) layout


A **ray** is half a line: it starts at a point and goes on forever in one
direction. ARENA writes every point on it as

> **P(u) = O + u · D**, with **u ≥ 0**

where **O** is the origin (a point), **D** the direction (a vector), and `u` a
scalar. `u = 0` is the origin, `u = 1` is one direction-length along, `u = 2`
twice that. Negative `u` would be *behind* the camera, so it is excluded.

In a tensor the ray is stored as a **(2, 3)** slab — row 0 is the origin, row
1 the direction, and the three columns are x, y, z:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']("import torch as t\n\nray = t.tensor([[0.0, 0.0, 0.0],    # O\n                [1.0, 0.5, 0.0]])   # D\norigin, direction = ray[0], ray[1]\nu = 2.0\nprint(origin + u * direction)       # the point at u = 2\n# Hidden checks\nassert _delta_output == 'tensor([2., 1., 0.])\\n'\n", globals()), end='')




`origin, direction = ray[0], ray[1]` unpacks a pair into two names.
The left and right sides have the same number of items, in the same order.

Because the whole thing is arithmetic, **many `u` at once** is one broadcast:
a column of `u` values times the direction row gives one point per row.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']("import torch as t\n\norigin = t.tensor([0.0, 0.0, 0.0])\ndirection = t.tensor([1.0, 0.5, 0.0])\nu = t.tensor([0.0, 1.0, 2.0])\npoints = origin + u[:, None] * direction    # (3,1) * (3,) → (3,3)\nprint(points)\n# Hidden checks\nassert _delta_output == 'tensor([[0.0000, 0.0000, 0.0000],\\n        [1.0000, 0.5000, 0.0000],\\n        [2.0000, 1.0000, 0.0000]])\\n'\n", globals()), end='')




Two facts you will use constantly in Chapter 0.1:

- The direction is **not unit length** and does not need to be. `D = (1, 0.5, 0)`
  is a perfectly good direction; `u` just measures in multiples of it.
- The same ray has many parametrisations — double `D` and halve every `u`
  and you get the same points. When ARENA asks "does the ray hit the
  segment", it solves for `u` and checks `u ≥ 0`, so **the sign of `u` is what
  carries the geometry**, not its size.

A family of rays all leaving the origin then differs only in the direction
row — which is exactly why `make_rays_1d` builds a stack of `(2, 3)` slabs and
only writes into row 1.


Where is the ray `O = (0, 0, 0)`, `D = (1, 2, 0)` when its x-coordinate is 3?
Solve `u · 1 = 3`, then read y = `u · 2`:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nray = t.tensor([[0.0, 0.0, 0.0], [1.0, 2.0, 0.0]])\nO, D = ray[0], ray[1]\nu = (3.0 - O[0]) / D[0]\npoint = O + u * D\nprint(u, point)\n# Hidden checks\nassert point.tolist() == [3.0, 6.0, 0.0]\n', globals()), end='')




| ray `[O, D]` | `u` | `O + u·D` |
| --- | --- | --- |
| `[[0,0,0],[1,0.5,0]]` | 0 | `[0, 0, 0]` |
| `[[0,0,0],[1,0.5,0]]` | 2 | `[2, 1, 0]` |
| `[[1,1,0],[0,1,0]]` | 3 | `[1, 4, 0]` |


<!-- dd:dd-q820 -->

### Problem 820 · faded — your turn

Row 0, row 1, then the one line of arithmetic.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[2.0, 1.0, 0.0]
```


In [ ]:
def solve(ray, u):
    """The point at parameter u on a ray stored as [origin, direction]."""
    ray = t.tensor(ray)
    origin, direction = ray[_____], ray[_____]
    return (origin + _____ * direction).tolist()


import torch as t
# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]], 2.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(820)


In [ ]:
#@title 💡 Solution — Problem 820
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(ray, u):
    """The point at parameter u on a ray stored as [origin, direction]."""
    ray = t.tensor(ray)
    origin, direction = ray[0], ray[1]
    return (origin + u * direction).tolist()


example = ([[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]], 2.0)
print(solve(*example))


Where is the ray `O = (0, 0, 0)`, `D = (1, 2, 0)` when its x-coordinate is 3?
Solve `u · 1 = 3`, then read y = `u · 2`:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nray = t.tensor([[0.0, 0.0, 0.0], [1.0, 2.0, 0.0]])\nO, D = ray[0], ray[1]\nu = (3.0 - O[0]) / D[0]\npoint = O + u * D\nprint(u, point)\n# Hidden checks\nassert point.tolist() == [3.0, 6.0, 0.0]\n', globals()), end='')




| ray `[O, D]` | `u` | `O + u·D` |
| --- | --- | --- |
| `[[0,0,0],[1,0.5,0]]` | 0 | `[0, 0, 0]` |
| `[[0,0,0],[1,0.5,0]]` | 2 | `[2, 1, 0]` |
| `[[1,1,0],[0,1,0]]` | 3 | `[1, 4, 0]` |


<!-- dd:dd-q821 -->

### Problem 821 · faded — your turn

u needs a trailing axis so it lines up against the 3 coordinates.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0.0, 0.0, 0.0], [1.0, 0.5, 0.0], [2.0, 1.0, 0.0]]
```


In [ ]:
def solve(ray, us):
    """Many points on one ray, by broadcasting u down a column."""
    ray = t.tensor(ray)
    u = t.tensor(us)
    return (ray[0] + u[_____, _____] * ray[1]).tolist()


import torch as t
# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]], [0.0, 1.0, 2.0])
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(821)


In [ ]:
#@title 💡 Solution — Problem 821
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(ray, us):
    """Many points on one ray, by broadcasting u down a column."""
    ray = t.tensor(ray)
    u = t.tensor(us)
    return (ray[0] + u[:, None] * ray[1]).tolist()


example = ([[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]], [0.0, 1.0, 2.0])
print(solve(*example))


<!-- dd:dd-q822 -->

### Problem 822 · independent

Return the parameter `u` at which the ray `[[ox, oy, oz], [dx, dy, dz]]` reaches x-coordinate `x`, as a float. Assume `dx != 0`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
2.0
```


In [ ]:
import torch as t


def solve(ray, x):
    """Solve O_x + u·D_x = x for u."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[1.0, 0.0, 0.0], [2.0, 0.0, 0.0]], 5.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(822)


In [ ]:
#@title 💡 Solution — Problem 822
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(ray, x):
    """Solve O_x + u·D_x = x for u."""
    ray = t.tensor(ray)
    origin, direction = ray[0], ray[1]
    return float((x - origin[0]) / direction[0])


example = ([[1.0, 0.0, 0.0], [2.0, 0.0, 0.0]], 5.0)
print(solve(*example))


<!-- dd:dd-q823 -->

### Problem 823 · independent

Return the y-coordinate (float) of the ray `[[ox, oy, oz], [dx, dy, dz]]` at the point where its x-coordinate equals `x`. Assume `dx != 0`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
6.0
```


In [ ]:
import torch as t


def solve(ray, x):
    """Solve for u from x, then read y."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[0.0, 0.0, 0.0], [1.0, 2.0, 0.0]], 3.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(823)


In [ ]:
#@title 💡 Solution — Problem 823
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(ray, x):
    """Solve for u from x, then read y."""
    ray = t.tensor(ray)
    u = (x - ray[0, 0]) / ray[1, 0]
    return float(ray[0, 1] + u * ray[1, 1])


example = ([[0.0, 0.0, 0.0], [1.0, 2.0, 0.0]], 3.0)
print(solve(*example))


<!-- dd:dd-q824 -->

### Problem 824 · independent

Return `True` if the point `p` (list of three floats) with x-coordinate `p[0]` lies on the FORWARD half of the ray `[[ox, oy, oz], [dx, dy, dz]]`, i.e. the `u` that reaches `p[0]` is ≥ 0 AND `O + u·D` equals `p` exactly. Assume `dx != 0`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
False
```


In [ ]:
import torch as t


def solve(ray, p):
    """Is p ahead of the camera on this ray?"""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[0.0, 0.0, 0.0], [1.0, 2.0, 0.0]], [-1.0, -2.0, 0.0])
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(824)


In [ ]:
#@title 💡 Solution — Problem 824
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(ray, p):
    """Is p ahead of the camera on this ray?"""
    ray = t.tensor(ray)
    u = (p[0] - ray[0, 0]) / ray[1, 0]
    on_line = (ray[0] + u * ray[1]).tolist() == p
    return bool(u >= 0) and on_line


example = ([[0.0, 0.0, 0.0], [1.0, 2.0, 0.0]], [-1.0, -2.0, 0.0])
print(solve(*example))


<!-- dd:dd-q825 -->

### Problem 825 · independent

Return the point at parameter `u` on the ray `[[ox, oy, oz], [dx, dy, dz]]` and on the same ray with its direction doubled and `u` halved, as a tuple of two lists. They must agree — the parametrisation is not unique.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
([2.0, 1.0, 0.0], [2.0, 1.0, 0.0])
```


In [ ]:
import torch as t


def solve(ray, u):
    """Doubling D and halving u names the same point."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]], 2.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(825)


In [ ]:
#@title 💡 Solution — Problem 825
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(ray, u):
    """Doubling D and halving u names the same point."""
    ray = t.tensor(ray)
    a = ray[0] + u * ray[1]
    b = ray[0] + (u / 2) * (2 * ray[1])
    return (a.tolist(), b.tolist())


example = ([[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]], 2.0)
print(solve(*example))


<!-- dd:dd-q826 -->

### Problem 826 · independent

Return the points at parameters `0, 1, …, n-1` on the ray `[[ox, oy, oz], [dx, dy, dz]]` as a nested list, using `t.arange(n)` and one broadcast.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[0.0, 0.0, 0.0], [1.0, 0.5, 0.0], [2.0, 1.0, 0.0]]
```


In [ ]:
import torch as t


def solve(ray, n):
    """n integer steps along one ray."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]], 3)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(826)


In [ ]:
#@title 💡 Solution — Problem 826
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(ray, n):
    """n integer steps along one ray."""
    ray = t.tensor(ray)
    u = t.arange(n)[:, None]
    return (ray[0] + u * ray[1]).tolist()


example = ([[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]], 3)
print(solve(*example))


<!-- dd:dd-q827 -->

### Problem 827 · independent

Given a stack of rays as a nested list of shape (n, 2, 3) and one scalar `u`, return the n points `O_i + u·D_i` as a nested list — index the origin rows and direction rows of every ray at once.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[2.0, 1.0, 0.0], [2.0, -1.0, 0.0]]
```


In [ ]:
import torch as t


def solve(rays, u):
    """One u, many rays."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = ([[[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]], [[0.0, 0.0, 0.0], [1.0, -0.5, 0.0]]], 2.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(827)


In [ ]:
#@title 💡 Solution — Problem 827
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(rays, u):
    """One u, many rays."""
    rays = t.tensor(rays)
    return (rays[:, 0] + u * rays[:, 1]).tolist()


example = ([[[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]], [[0.0, 0.0, 0.0], [1.0, -0.5, 0.0]]], 2.0)
print(solve(*example))


#### Common mistakes

- **"The direction must be a unit vector."** No. `u` is measured in
  direction-lengths; any nonzero `D` is fine, and ARENA's rays are not
  normalised.
- **"Solving for the line is solving for the ray."** A line accepts any
  `u`; a ray needs `u ≥ 0`. Every "does this ray hit" test in 0.1 ends with
  that sign check.
- **"`rays[0]` is the origins."** For a stack of rays `rays[0]` is the whole
  first ray. The origins of every ray are `rays[:, 0]`.


<!-- dd:dd-kp-raytracing-make-rays-1d -->

## make_rays_1d: a fan of rays as one (n, 2, 3) tensor

`raytracing.make-rays-1d`


### zeros, then write the two direction columns


ARENA's first exercise, `make_rays_1d(num_pixels, y_limit)`, asks for a
**fan** of rays: all from the origin, all with direction x = 1, and direction
y spread evenly from `-y_limit` to `+y_limit`. The result is one tensor of
shape **`(num_pixels, 2, 3)`** — one `(2, 3)` ray slab per pixel.

Everything you need is the three pages before this one, in order:

1. **Canvas.** `t.zeros((num_pixels, 2, 3))` — every origin is already
   `(0, 0, 0)` and every direction z is already `0`. Two columns remain.
2. **Direction x.** Every ray's `[1, 0]` slot is `1`: a scalar broadcast
   through a slice, `rays[:, 1, 0] = 1`.
3. **Direction y.** `num_pixels` values from `-y_limit` to `y_limit`, written
   into the `[:, 1, 1]` column — `t.linspace(..., out=rays[:, 1, 1])`, or the
   same with assignment.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']("import torch as t\n\ndef make_rays_1d(num_pixels, y_limit):\n    rays = t.zeros((num_pixels, 2, 3), dtype=t.float32)\n    t.linspace(-y_limit, y_limit, num_pixels, out=rays[:, 1, 1])\n    rays[:, 1, 0] = 1\n    return rays\n\nprint(make_rays_1d(3, 1.0))\n# Hidden checks\nassert _delta_output == 'tensor([[[ 0.,  0.,  0.],\\n         [ 1., -1.,  0.]],\\n\\n        [[ 0.,  0.,  0.],\\n         [ 1.,  0.,  0.]],\\n\\n        [[ 0.,  0.,  0.],\\n         [ 1.,  1.,  0.]]])\\n'\n", globals()), end='')




Read the output as three slabs. Row 0 of each is the origin; row 1 is
`(1, y, 0)` with `y` walking `-1, 0, 1`. Nothing else was touched.

The habit to take from this: **allocate the final shape once, then write the
parts that vary**. Building rows in a loop and stacking them gives the same
numbers and is the thing you will unlearn in the next chapter.


The same construction with y spread over `[-2, 2]` in 5 steps, checked
against the two facts that define it:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nrays = t.zeros((5, 2, 3))\nrays[:, 1, 0] = 1\nrays[:, 1, 1] = t.linspace(-2.0, 2.0, 5)\nprint(rays[:, 1])                          # every direction row\n# Hidden checks\nassert (rays[:, 0] == 0).all()             # all origins at 0\nassert rays[:, 1, 1].tolist() == [-2.0, -1.0, 0.0, 1.0, 2.0]\n', globals()), end='')




| `num_pixels, y_limit` | direction rows `rays[:, 1]` |
| --- | --- |
| `2, 1.0` | `[[1, -1, 0], [1, 1, 0]]` |
| `3, 3.0` | `[[1, -3, 0], [1, 0, 0], [1, 3, 0]]` |


<!-- dd:dd-q831 -->

### Problem 831 · faded — your turn

Which row is the direction, which column is x, and the two ends of the fan.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[[0.0, 0.0, 0.0], [1.0, -1.0, 0.0]], [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [1.0, 1.0, 0.0]]]
```


In [ ]:
def solve(num_pixels, y_limit):
    """The fan of rays: zeros, then two direction columns."""
    rays = t.zeros((num_pixels, 2, 3))
    rays[_____, _____, _____] = 1
    t.linspace(_____, _____, num_pixels, out=rays[:, 1, 1])
    return rays.tolist()


import torch as t
# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 1.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(831)


In [ ]:
#@title 💡 Solution — Problem 831
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(num_pixels, y_limit):
    """The fan of rays: zeros, then two direction columns."""
    rays = t.zeros((num_pixels, 2, 3))
    rays[:, 1, 0] = 1
    t.linspace(-y_limit, y_limit, num_pixels, out=rays[:, 1, 1])
    return rays.tolist()


example = (3, 1.0)
print(solve(*example))


The same construction with y spread over `[-2, 2]` in 5 steps, checked
against the two facts that define it:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\n\nrays = t.zeros((5, 2, 3))\nrays[:, 1, 0] = 1\nrays[:, 1, 1] = t.linspace(-2.0, 2.0, 5)\nprint(rays[:, 1])                          # every direction row\n# Hidden checks\nassert (rays[:, 0] == 0).all()             # all origins at 0\nassert rays[:, 1, 1].tolist() == [-2.0, -1.0, 0.0, 1.0, 2.0]\n', globals()), end='')




| `num_pixels, y_limit` | direction rows `rays[:, 1]` |
| --- | --- |
| `2, 1.0` | `[[1, -1, 0], [1, 1, 0]]` |
| `3, 3.0` | `[[1, -3, 0], [1, 0, 0], [1, 3, 0]]` |


<!-- dd:dd-q832 -->

### Problem 832 · faded — your turn

Same slots as before; only the spelling of the write changes.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[[0.0, 0.0, 0.0], [1.0, -1.0, 0.0]], [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [1.0, 1.0, 0.0]]]
```


In [ ]:
def solve(num_pixels, y_limit):
    """make_rays_1d with the y column assigned."""
    rays = t.zeros((num_pixels, 2, 3))
    rays[:, 1, 0] = 1
    rays[_____, _____, _____] = t.linspace(-y_limit, y_limit, num_pixels)
    return rays.tolist()


import torch as t
# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 1.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(832)


In [ ]:
#@title 💡 Solution — Problem 832
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(num_pixels, y_limit):
    """make_rays_1d with the y column assigned."""
    rays = t.zeros((num_pixels, 2, 3))
    rays[:, 1, 0] = 1
    rays[:, 1, 1] = t.linspace(-y_limit, y_limit, num_pixels)
    return rays.tolist()


example = (3, 1.0)
print(solve(*example))


<!-- dd:dd-q833 -->

### Problem 833 · independent

Return only the direction rows of `make_rays_1d(num_pixels, y_limit)` — a `(num_pixels, 3)` nested list of `[1, y, 0]` rows with `y` from `-y_limit` to `y_limit`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[1.0, -1.0, 0.0], [1.0, 0.0, 0.0], [1.0, 1.0, 0.0]]
```


In [ ]:
import torch as t


def solve(num_pixels, y_limit):
    """The direction rows of the fan."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 1.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(833)


In [ ]:
#@title 💡 Solution — Problem 833
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(num_pixels, y_limit):
    """The direction rows of the fan."""
    rays = t.zeros((num_pixels, 2, 3))
    rays[:, 1, 0] = 1
    rays[:, 1, 1] = t.linspace(-y_limit, y_limit, num_pixels)
    return rays[:, 1].tolist()


example = (3, 1.0)
print(solve(*example))


<!-- dd:dd-q834 -->

### Problem 834 · independent

A fan in the z direction instead of y: `(num_pixels, 2, 3)` rays with direction `[1, 0, z]`, `z` evenly from `-z_limit` to `z_limit`. Return as a nested list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[[0.0, 0.0, 0.0], [1.0, 0.0, -1.0]], [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [1.0, 0.0, 1.0]]]
```


In [ ]:
import torch as t


def solve(num_pixels, z_limit):
    """make_rays_1d rotated into the x–z plane."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 1.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(834)


In [ ]:
#@title 💡 Solution — Problem 834
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(num_pixels, z_limit):
    """make_rays_1d rotated into the x–z plane."""
    rays = t.zeros((num_pixels, 2, 3))
    rays[:, 1, 0] = 1
    rays[:, 1, 2] = t.linspace(-z_limit, z_limit, num_pixels)
    return rays.tolist()


example = (3, 1.0)
print(solve(*example))


<!-- dd:dd-q835 -->

### Problem 835 · independent

A fan whose rays all start at the point `origin` (list of three floats) instead of zero: build `make_rays_1d(num_pixels, y_limit)` and then write the origin row of every ray in one broadcast. Return as a nested list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[[0.0, 0.0, 1.0], [1.0, -1.0, 0.0]], [[0.0, 0.0, 1.0], [1.0, 1.0, 0.0]]]
```


In [ ]:
import torch as t


def solve(num_pixels, y_limit, origin):
    """The fan, moved to a shared non-zero origin."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (2, 1.0, [0.0, 0.0, 1.0])
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(835)


In [ ]:
#@title 💡 Solution — Problem 835
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(num_pixels, y_limit, origin):
    """The fan, moved to a shared non-zero origin."""
    rays = t.zeros((num_pixels, 2, 3))
    rays[:, 0] = t.tensor(origin)
    rays[:, 1, 0] = 1
    rays[:, 1, 1] = t.linspace(-y_limit, y_limit, num_pixels)
    return rays.tolist()


example = (2, 1.0, [0.0, 0.0, 1.0])
print(solve(*example))


<!-- dd:dd-q836 -->

### Problem 836 · independent

Return `(shape, all_origins_zero, xs)` for `make_rays_1d(num_pixels, y_limit)`: its shape as a tuple, whether every origin entry is 0 (a bool), and the list of direction-x values.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
((3, 2, 3), True, [1.0, 1.0, 1.0])
```


In [ ]:
import torch as t


def solve(num_pixels, y_limit):
    """Three checks on a freshly built fan."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 1.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(836)


In [ ]:
#@title 💡 Solution — Problem 836
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(num_pixels, y_limit):
    """Three checks on a freshly built fan."""
    rays = t.zeros((num_pixels, 2, 3))
    rays[:, 1, 0] = 1
    rays[:, 1, 1] = t.linspace(-y_limit, y_limit, num_pixels)
    origins_zero = bool((rays[:, 0] == 0).all())
    return (tuple(rays.shape), origins_zero, rays[:, 1, 0].tolist())


example = (3, 1.0)
print(solve(*example))


<!-- dd:dd-q837 -->

### Problem 837 · independent

A fan where direction x is `x_dir` instead of 1 (the rays still fan from `-y_limit` to `y_limit` in y). Return `make_rays_1d`-shaped output as a nested list.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[[[0.0, 0.0, 0.0], [2.0, -1.0, 0.0]], [[0.0, 0.0, 0.0], [2.0, 0.0, 0.0]], [[0.0, 0.0, 0.0], [2.0, 1.0, 0.0]]]
```


In [ ]:
import torch as t


def solve(num_pixels, y_limit, x_dir):
    """The same fan with a different x speed."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 1.0, 2.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(837)


In [ ]:
#@title 💡 Solution — Problem 837
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(num_pixels, y_limit, x_dir):
    """The same fan with a different x speed."""
    rays = t.zeros((num_pixels, 2, 3))
    rays[:, 1, 0] = x_dir
    rays[:, 1, 1] = t.linspace(-y_limit, y_limit, num_pixels)
    return rays.tolist()


example = (3, 1.0, 2.0)
print(solve(*example))


<!-- dd:dd-q838 -->

### Problem 838 · independent

Return the y-coordinate of every ray of `make_rays_1d(num_pixels, y_limit)` at x = `x` as a list of floats. Since every direction has x = 1 and every origin is 0, `u` at x is simply `x`.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
[-2.0, 0.0, 2.0]
```


In [ ]:
import torch as t


def solve(num_pixels, y_limit, x):
    """Where the fan is at a given x."""
    return None

# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 1.0, 2.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(838)


In [ ]:
#@title 💡 Solution — Problem 838
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t


def solve(num_pixels, y_limit, x):
    """Where the fan is at a given x."""
    rays = t.zeros((num_pixels, 2, 3))
    rays[:, 1, 0] = 1
    rays[:, 1, 1] = t.linspace(-y_limit, y_limit, num_pixels)
    points = rays[:, 0] + x * rays[:, 1]
    return points[:, 1].tolist()


example = (3, 1.0, 2.0)
print(solve(*example))


#### Common mistakes

- **"Build each ray, then stack."** It gives the same numbers, but the
  exercise is teaching the allocate-then-write habit that every later
  chapter uses. One canvas, two column writes.
- **"Row 0 / row 1 are x / y."** Rows are origin and direction; the *columns*
  are x, y, z. `rays[:, 1, 1]` is direction-y for every ray.
- **"`linspace(0, y_limit)`."** The fan is symmetric: from `-y_limit` to
  `+y_limit`.
